# R.O.A.D. Barbados: PP-OCRv6 medium (kraken) fine-tune, canonical fold 0, free Colab T4

Run in order on a T4 runtime. Inputs come from your private Drive `road/`; nothing is uploaded.
Hypothesis: Fine-tuning the Apache-2.0 PP-OCRv6 medium CTC line recognizer (kraken 7.1.1) on the fold-0 training rows gives fold-0 combined error below the TrOCR control 0.15076, and/or adds a diverse vote member (gain >= 0.002, 95% bootstrap low bound > 0).
Controls: same frozen folds (checksum-verified), seed 20260906, no test-set use for training. Weights: Apache-2.0.
Hyper-parameters (AdamW lr 0.0002, 12 epochs, batch 8, cosine, warmup 100) are a first guess, not tuned.
Stop: smoke forecast above the 9 h gate, nonfinite loss, or fold-0 combined above the control.
Caveat: PP-OCRv6 training drops lines wider than 2560 px at height 96 (about 2% of rows).

In [ ]:
# Configuration: edit before Run All.
DATA_ROOT = "/content/drive/MyDrive/road"   # Train.csv, Test.csv, images/ (or images.zip); private, never uploaded
OUTPUT_ROOT = "/content/drive/MyDrive/road/results"
FOLD, SEED = 0, 20260906
EPOCHS, LRATE, BATCH, PRECISION = 12, 0.0002, 8, "16-mixed"
MAX_ESTIMATED_HOURS = 9.0
WEIGHTS_URL, WEIGHTS_SHA = "https://zenodo.org/records/21788410/files/medium.safetensors", "15313b51ace64cbfa81f8f6ef25ad64f04e5a6fb7f7823e67b107527bc081ac9"
FOLDS_SHA = "fff8a4994bc606cdd9987ae521712976bd83c180223dce596340d192ff58a448"   # sha256 of the frozen data/splits/folds.csv; regenerated fold file must match
SOURCE_FILES = {"src/road_ocr/__init__.py": "\"\"\"Utilities for reproducible Barbados handwriting experiments.\"\"\"\n\nfrom .metrics import Score, score_pairs\n\n__all__ = [\"Score\", \"score_pairs\"]\n", "src/road_ocr/metrics.py": "\"\"\"Dependency-free OCR error metrics.\n\nThe competition page says longer references receive more weight. The local\nimplementation therefore uses corpus (micro-averaged) error rates: summed edit\ndistance divided by summed reference length. Keep this implementation versioned\nand do not silently normalize text before scoring.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Iterable, Sequence, TypeVar\n\nT = TypeVar(\"T\")\n\n\ndef edit_distance(reference: Sequence[T], prediction: Sequence[T]) -> int:\n    \"\"\"Return Levenshtein distance using O(min(n, m)) memory.\"\"\"\n\n    if len(reference) < len(prediction):\n        reference, prediction = prediction, reference\n    previous = list(range(len(prediction) + 1))\n    for i, ref_item in enumerate(reference, start=1):\n        current = [i]\n        for j, pred_item in enumerate(prediction, start=1):\n            current.append(\n                min(\n                    current[-1] + 1,\n                    previous[j] + 1,\n                    previous[j - 1] + (ref_item != pred_item),\n                )\n            )\n        previous = current\n    return previous[-1]\n\n\ndef combined_error(reference: str, prediction: str) -> float:\n    \"\"\"The scorer's per-line 0.5/0.5 error, for ranking candidates within one row.\n\n    Corpus scores stay micro-averaged through ``score_pairs``; this is the\n    per-line view a preference pair or a reward needs.\n    \"\"\"\n\n    reference_tokens, prediction_tokens = reference.split(), prediction.split()\n    if not reference or not reference_tokens:\n        raise ValueError(\"reference line is empty\")\n    cer = edit_distance(reference, prediction) / len(reference)\n    wer = edit_distance(reference_tokens, prediction_tokens) / len(reference_tokens)\n    return 0.5 * cer + 0.5 * wer\n\n\n@dataclass(frozen=True)\nclass Score:\n    cer: float\n    wer: float\n    combined: float\n    character_edits: int\n    reference_characters: int\n    word_edits: int\n    reference_words: int\n    samples: int\n\n\ndef score_pairs(pairs: Iterable[tuple[str, str]]) -> Score:\n    \"\"\"Compute micro CER/WER and the official 0.5/0.5 combined score.\"\"\"\n\n    character_edits = 0\n    reference_characters = 0\n    word_edits = 0\n    reference_words = 0\n    samples = 0\n\n    for reference, prediction in pairs:\n        if not isinstance(reference, str) or not isinstance(prediction, str):\n            raise TypeError(\"reference and prediction values must be strings\")\n        reference_tokens = reference.split()\n        prediction_tokens = prediction.split()\n        character_edits += edit_distance(reference, prediction)\n        reference_characters += len(reference)\n        word_edits += edit_distance(reference_tokens, prediction_tokens)\n        reference_words += len(reference_tokens)\n        samples += 1\n\n    if samples == 0:\n        raise ValueError(\"cannot score an empty collection\")\n    if reference_characters == 0 or reference_words == 0:\n        raise ValueError(\"reference corpus must contain characters and words\")\n\n    cer = character_edits / reference_characters\n    wer = word_edits / reference_words\n    return Score(\n        cer=cer,\n        wer=wer,\n        combined=0.5 * cer + 0.5 * wer,\n        character_edits=character_edits,\n        reference_characters=reference_characters,\n        word_edits=word_edits,\n        reference_words=reference_words,\n        samples=samples,\n    )\n", "src/road_ocr/records.py": "\"\"\"Strict CSV readers shared by command-line checks.\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nfrom pathlib import Path\n\n\nclass CsvValidationError(ValueError):\n    pass\n\n\ndef read_csv(path: str | Path, expected_columns: list[str]) -> list[dict[str, str]]:\n    path = Path(path)\n    with path.open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != expected_columns:\n            raise CsvValidationError(\n                f\"{path}: expected columns {expected_columns}, got {reader.fieldnames}\"\n            )\n        rows = list(reader)\n    return rows\n\n\ndef index_unique(rows: list[dict[str, str]], path: str | Path) -> dict[str, dict[str, str]]:\n    indexed: dict[str, dict[str, str]] = {}\n    for row_number, row in enumerate(rows, start=2):\n        record_id = row.get(\"ID\", \"\")\n        if not record_id:\n            raise CsvValidationError(f\"{path}:{row_number}: empty ID\")\n        if record_id in indexed:\n            raise CsvValidationError(f\"{path}:{row_number}: duplicate ID {record_id!r}\")\n        indexed[record_id] = row\n    return indexed\n", "scripts/make_folds.py": "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom road_ocr.records import index_unique, read_csv\n\n\ndef stable_tiebreaker(seed: int, value: str) -> str:\n    return hashlib.sha256(f\"{seed}:{value}\".encode()).hexdigest()\n\n\ndef assign_folds(rows: list[dict[str, str]], folds: int, seed: int) -> dict[str, int]:\n    \"\"\"Keep exact duplicate labels together while balancing rows and characters.\"\"\"\n\n    groups: dict[str, list[dict[str, str]]] = defaultdict(list)\n    for row in rows:\n        groups[row[\"Target\"]].append(row)\n\n    ordered_groups = sorted(\n        groups.items(),\n        key=lambda item: (\n            -sum(len(row[\"Target\"]) for row in item[1]),\n            -len(item[1]),\n            stable_tiebreaker(seed, item[0]),\n        ),\n    )\n    fold_rows = [0] * folds\n    fold_characters = [0] * folds\n    assignments: dict[str, int] = {}\n\n    for _label, group in ordered_groups:\n        fold = min(range(folds), key=lambda i: (fold_characters[i], fold_rows[i], i))\n        for row in group:\n            assignments[row[\"ID\"]] = fold\n        fold_rows[fold] += len(group)\n        fold_characters[fold] += sum(len(row[\"Target\"]) for row in group)\n    return assignments\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Create deterministic grouped OCR folds.\")\n    parser.add_argument(\"--train\", default=\"Train.csv\")\n    parser.add_argument(\"--output\", default=\"data/splits/folds.csv\")\n    parser.add_argument(\"--folds\", type=int, default=5)\n    parser.add_argument(\"--seed\", type=int, default=20260906)\n    args = parser.parse_args()\n    if args.folds < 2:\n        parser.error(\"--folds must be at least 2\")\n\n    rows = read_csv(args.train, [\"ID\", \"Target\"])\n    index_unique(rows, args.train)\n    assignments = assign_folds(rows, args.folds, args.seed)\n\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with output.open(\"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"ID\", \"fold\"])\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({\"ID\": row[\"ID\"], \"fold\": assignments[row[\"ID\"]]})\n\n    counts = [sum(fold == i for fold in assignments.values()) for i in range(args.folds)]\n    print(f\"wrote {len(assignments)} assignments to {output}\")\n    print(f\"fold row counts: {counts}; seed={args.seed}\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/prepare_kraken_data.py": "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nfrom pathlib import Path\n\nfrom road_ocr.records import CsvValidationError, index_unique, read_csv\n\n\ndef stable_order(seed: int, record_id: str) -> str:\n    return hashlib.sha256(f\"{seed}:{record_id}\".encode()).hexdigest()\n\n\ndef select_rows(\n    rows: list[dict[str, str]],\n    fold_by_id: dict[str, int],\n    validation_fold: int,\n    seed: int,\n    train_limit: int | None,\n    validation_limit: int | None,\n) -> tuple[list[dict[str, str]], list[dict[str, str]]]:\n    train = [row for row in rows if fold_by_id[row[\"ID\"]] != validation_fold]\n    validation = [row for row in rows if fold_by_id[row[\"ID\"]] == validation_fold]\n    train.sort(key=lambda row: stable_order(seed, row[\"ID\"]))\n    validation.sort(key=lambda row: stable_order(seed, row[\"ID\"]))\n    if train_limit is not None:\n        train = train[:train_limit]\n    if validation_limit is not None:\n        validation = validation[:validation_limit]\n    return train, validation\n\n\ndef materialize(\n    rows: list[dict[str, str]], image_dir: Path, output_dir: Path, manifest: Path\n) -> None:\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with manifest.open(\"w\", encoding=\"utf-8\") as handle:\n        for row in rows:\n            source = image_dir / f\"{row['ID']}.jpg\"\n            if not source.is_file():\n                raise FileNotFoundError(source)\n            target = output_dir / source.name\n            if not target.exists():\n                try:\n                    target.symlink_to(source.resolve())\n                except OSError:\n                    target.write_bytes(source.read_bytes())\n            label_path = target.with_suffix(\".gt.txt\")\n            label_path.write_text(row[\"Target\"], encoding=\"utf-8\")\n            manifest_path = target.absolute()\n            if not manifest_path.with_suffix(\".gt.txt\").is_file():\n                raise FileNotFoundError(manifest_path.with_suffix(\".gt.txt\"))\n            handle.write(f\"{manifest_path}\\n\")\n\n\ndef write_labels(rows: list[dict[str, str]], path: Path) -> None:\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"ID\", \"Target\"])\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef prepare_run_directories(data_output: Path) -> None:\n    \"\"\"Create the data directory and Kraken's non-creating output parent.\"\"\"\n    data_output.mkdir(parents=True, exist_ok=True)\n    (data_output.parent / \"checkpoints\").mkdir(parents=True, exist_ok=True)\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Prepare fixed-fold Kraken path data.\")\n    parser.add_argument(\"--train\", default=\"Train.csv\")\n    parser.add_argument(\"--images\", default=\"images\")\n    parser.add_argument(\"--fold-manifest\", default=\"data/splits/folds.csv\")\n    parser.add_argument(\"--fold\", type=int, required=True)\n    parser.add_argument(\"--seed\", type=int, default=20260906)\n    parser.add_argument(\"--train-limit\", type=int)\n    parser.add_argument(\"--validation-limit\", type=int)\n    parser.add_argument(\"--output\", required=True)\n    args = parser.parse_args()\n\n    rows = read_csv(args.train, [\"ID\", \"Target\"])\n    index_unique(rows, args.train)\n    with Path(args.fold_manifest).open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != [\"ID\", \"fold\"]:\n            raise CsvValidationError(\n                f\"{args.fold_manifest}: expected columns ['ID', 'fold'], got {reader.fieldnames}\"\n            )\n        fold_by_id = {row[\"ID\"]: int(row[\"fold\"]) for row in reader}\n    missing = [row[\"ID\"] for row in rows if row[\"ID\"] not in fold_by_id]\n    if missing:\n        raise CsvValidationError(f\"fold manifest is missing {len(missing)} IDs\")\n\n    train, validation = select_rows(\n        rows,\n        fold_by_id,\n        args.fold,\n        args.seed,\n        args.train_limit,\n        args.validation_limit,\n    )\n    output = Path(args.output)\n    prepare_run_directories(output)\n    materialize(train, Path(args.images), output / \"train\", output / \"train.txt\")\n    materialize(\n        validation,\n        Path(args.images),\n        output / \"validation\",\n        output / \"validation.txt\",\n    )\n    write_labels(train, output / \"train_labels.csv\")\n    write_labels(validation, output / \"validation_labels.csv\")\n    print(\n        f\"prepared train={len(train)} validation={len(validation)} \"\n        f\"for fold={args.fold} under {output}\"\n    )\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n", "scripts/kraken_infer.py": "#!/usr/bin/env python3\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport shlex\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom road_ocr.metrics import score_pairs\nfrom road_ocr.records import CsvValidationError, index_unique, read_csv\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_fold_ids(path: str | Path, fold: int) -> set[str]:\n    with Path(path).open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as handle:\n        reader = csv.DictReader(handle)\n        if reader.fieldnames != [\"ID\", \"fold\"]:\n            raise CsvValidationError(\n                f\"{path}: expected columns ['ID', 'fold'], got {reader.fieldnames}\"\n            )\n        return {row[\"ID\"] for row in reader if int(row[\"fold\"]) == fold}\n\n\ndef make_full_crop_segmentation(image, image_path: Path):\n    from kraken.containers import BaselineLine, Segmentation\n\n    width, height = image.size\n    right = max(0, width - 1)\n    bottom = max(0, height - 1)\n    baseline_y = max(0, height - 2)\n    line = BaselineLine(\n        id=image_path.stem,\n        baseline=[(0, baseline_y), (right, baseline_y)],\n        boundary=[(0, 0), (right, 0), (right, bottom), (0, bottom), (0, 0)],\n        base_dir=\"L\",\n    )\n    return Segmentation(\n        type=\"baselines\",\n        imagename=image_path,\n        text_direction=\"horizontal-lr\",\n        script_detection=False,\n        lines=[line],\n    )\n\n\ndef predict_rows(rows, image_dir: Path, model_path: Path, device: str, pad: int):\n    from PIL import Image\n    from kraken.lib.models import load_any\n    from kraken.rpred import rpred\n\n    new_api = model_path.suffix == \".safetensors\"  # kraken >= 7 ppocrv6 weights; load_any is legacy VGSL only\n    if new_api:\n        from kraken.configs import RecognitionInferenceConfig\n        from kraken.tasks import RecognitionTaskModel\n\n        task = RecognitionTaskModel.load_model(model_path)\n        config = RecognitionInferenceConfig(accelerator=\"cpu\" if device == \"cpu\" else \"auto\")\n    else:\n        model = load_any(model_path, device=device)\n    predictions: list[dict[str, str]] = []\n    for index, row in enumerate(rows, start=1):\n        image_path = image_dir / f\"{row['ID']}.jpg\"\n        if not image_path.is_file():\n            raise FileNotFoundError(image_path)\n        with Image.open(image_path) as image:\n            segmentation = make_full_crop_segmentation(image, image_path)\n            records = (\n                task.predict(image, segmentation, config)\n                if new_api\n                else rpred(\n                    model,\n                    image,\n                    segmentation,\n                    pad=pad,\n                    bidi_reordering=True,\n                    no_legacy_polygons=True,\n                )\n            )\n            prediction = \" \".join(record.prediction for record in records)\n        prediction = \" \".join(prediction.split())\n        predictions.append({\"ID\": row[\"ID\"], \"Target\": prediction})\n        if index == 1 or index % 25 == 0 or index == len(rows):\n            print(f\"recognized {index}/{len(rows)}\", flush=True)\n    return predictions\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=\"Run a Kraken baseline reproducibly.\")\n    parser.add_argument(\"--input\", required=True, help=\"ID or ID,Target CSV\")\n    parser.add_argument(\"--images\", default=\"images\")\n    parser.add_argument(\"--model\", required=True)\n    parser.add_argument(\"--fold-manifest\")\n    parser.add_argument(\"--fold\", type=int)\n    parser.add_argument(\"--max-samples\", type=int)\n    parser.add_argument(\"--device\", default=\"cpu\")\n    parser.add_argument(\"--pad\", type=int, default=16)\n    parser.add_argument(\"--output\", required=True)\n    parser.add_argument(\"--metadata\")\n    args = parser.parse_args()\n    if (args.fold_manifest is None) != (args.fold is None):\n        parser.error(\"--fold-manifest and --fold must be provided together\")\n\n    with Path(args.input).open(\"r\", encoding=\"utf-8-sig\", newline=\"\") as handle:\n        fieldnames = csv.DictReader(handle).fieldnames\n    if fieldnames not in ([\"ID\"], [\"ID\", \"Target\"]):\n        parser.error(f\"{args.input}: expected ID or ID,Target columns, got {fieldnames}\")\n    rows = read_csv(args.input, fieldnames)\n    index_unique(rows, args.input)\n    if args.fold_manifest is not None:\n        fold_ids = read_fold_ids(args.fold_manifest, args.fold)\n        rows = [row for row in rows if row[\"ID\"] in fold_ids]\n    if args.max_samples is not None:\n        rows = rows[: args.max_samples]\n    if not rows:\n        parser.error(\"selection contains no rows\")\n\n    model_path = Path(args.model)\n    if not model_path.is_file():\n        parser.error(f\"model not found: {model_path}\")\n    output = Path(args.output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    started = time.time()\n    predictions = predict_rows(\n        rows, Path(args.images), model_path, device=args.device, pad=args.pad\n    )\n    runtime = time.time() - started\n    with output.open(\"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"ID\", \"Target\"])\n        writer.writeheader()\n        writer.writerows(predictions)\n\n    metadata = {\n        \"command\": \" \".join(shlex.quote(value) for value in sys.argv),\n        \"device\": args.device,\n        \"fold\": args.fold,\n        \"input\": args.input,\n        \"kraken_version\": importlib.metadata.version(\"kraken\"),\n        \"model\": str(model_path),\n        \"model_sha256\": sha256(model_path),\n        \"output\": str(output),\n        \"platform\": platform.platform(),\n        \"python\": platform.python_version(),\n        \"runtime_seconds\": runtime,\n        \"samples\": len(rows),\n        \"torch_version\": importlib.metadata.version(\"torch\"),\n    }\n    if fieldnames == [\"ID\", \"Target\"]:\n        result = score_pairs(\n            (row[\"Target\"], prediction[\"Target\"])\n            for row, prediction in zip(rows, predictions, strict=True)\n        )\n        metadata[\"score\"] = result.__dict__\n        print(json.dumps(result.__dict__, indent=2, sort_keys=True))\n    if args.metadata:\n        metadata_path = Path(args.metadata)\n        metadata_path.parent.mkdir(parents=True, exist_ok=True)\n        metadata_path.write_text(\n            json.dumps(metadata, indent=2, sort_keys=True) + \"\\n\", encoding=\"utf-8\"\n        )\n    print(f\"wrote {len(predictions)} predictions to {output} in {runtime:.1f}s\")\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"}
SOURCE_PROVENANCE = {"commit": "509af0b7b449a90c80e1fe7f84d1778ae786d26f", "branch": "exp/candidate-likelihood", "dirty_embedded_files": ["M scripts/kraken_infer.py"], "sha256": {"src/road_ocr/__init__.py": "dbce58ebcb86794db49293d6af1e52d1df39035d804d5237d5d7fbd76cf27309", "src/road_ocr/metrics.py": "95e4b16f677450b5d2f0325b2a404f744f54cb75da1b333f84c89ee23541c243", "src/road_ocr/records.py": "eefd65a36bba55a32d3ff2ae11e81b52103c83dfc351bfd974ecd7f9e0beab6f", "scripts/make_folds.py": "6b9c12cc1bee05f6e87dedda1ee371a51d324ad424fe215fcfd1db3467bc7491", "scripts/prepare_kraken_data.py": "f94d56bdb7143a270501ba93d994183269a5b816fda01c8fa2ff088560deb3af", "scripts/kraken_infer.py": "b5cedc53fbad8eff03a51109158f1f999e273956e0b412a150da5f717be25809"}}

In [ ]:
import os, sys, json, time, hashlib, shutil, subprocess, zipfile, uuid
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "Select a T4 GPU runtime"
print(torch.cuda.get_device_name(0))
RUN_ID = "colab_ppocrv6_medium_fold%d_%s_%s" % (FOLD, datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"), uuid.uuid4().hex[:8])
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
RUN = Path(OUTPUT_ROOT) / RUN_ID; RUN.mkdir(parents=True, exist_ok=False)
for rel, text in SOURCE_FILES.items():
    dest = WORK / rel; dest.parent.mkdir(parents=True, exist_ok=True); dest.write_text(text, encoding="utf-8")
(RUN / "source_provenance.json").write_text(json.dumps(SOURCE_PROVENANCE, indent=1))
data = Path(DATA_ROOT)
for name in ["Train.csv", "Test.csv"]:
    shutil.copy(data / name, WORK / name)
images = WORK / "images"
src_dir = next((p for p in data.rglob("images") if p.is_dir() and next(p.glob("*.jpg"), None)), None)
if src_dir:
    shutil.copytree(src_dir, images)
else:
    with zipfile.ZipFile(next(data.rglob("images.zip"))) as z:
        images.mkdir()
        for m in z.namelist():
            if m.lower().endswith(".jpg"):
                (images / Path(m).name).write_bytes(z.read(m))
print("images:", len(list(images.glob("*.jpg"))))
env = dict(os.environ, PYTHONPATH=str(WORK / "src"))
def sh(cmd, log=None, **kw):
    print("$", cmd, flush=True)
    with open(log or os.devnull, "w") as f:
        p = subprocess.Popen(cmd, shell=True, cwd=WORK, env={**env, **kw}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            f.write(line); f.flush(); print(line, end="")
        assert p.wait() == 0, "command failed"

In [ ]:
# Regenerate the frozen folds and require the repo checksum (never change folds to improve a score).
sh("python scripts/make_folds.py --train Train.csv --output data/splits/folds.csv")
got = hashlib.sha256((WORK / "data/splits/folds.csv").read_bytes()).hexdigest()
assert got == FOLDS_SHA, f"fold manifest mismatch: {got}"
print("folds verified")

In [ ]:
sh("pip install -q kraken==7.1.1")
# kraken 7.1.1 pins safetensors~=0.7.0, but only uses the stable save_file/load_file/safe_open
# API; something else in this image needs safetensors>=0.8.0 at import time (transformers'
# own version check). --no-deps skips re-checking kraken's declared range for this one upgrade.
sh("pip install -q --no-deps -U 'safetensors>=0.8.0'")
import importlib.metadata as md
print("kraken", md.version("kraken"), "torch", md.version("torch"))
import importlib; importlib.invalidate_caches()
import torch; assert torch.cuda.is_available(), "pip replaced torch with a non-CUDA build"
sh(f"curl -sL -o /content/medium.safetensors {WEIGHTS_URL}")
got = hashlib.sha256(Path("/content/medium.safetensors").read_bytes()).hexdigest()
assert got == WEIGHTS_SHA, f"weights sha mismatch {got}"
sh(f"python scripts/prepare_kraken_data.py --fold {FOLD} --seed 20260906 --output /content/work/kdata")

In [ ]:
# Smoke: 32 rows, 1 epoch. Measures seconds/step and gates the full run by forecast hours.
sh("head -32 kdata/train.txt > kdata/smoke_train.txt; head -8 kdata/validation.txt > kdata/smoke_val.txt")
t0 = time.time()
sh(("ketos -d cuda:0 --precision %s --workers 2 --threads 2 -s %d train --arch ppocrv6 --variant medium "
    "-i /content/medium.safetensors --resize union -f path -t kdata/smoke_train.txt -e kdata/smoke_val.txt "
    "-B %d -r %g --optimizer AdamW --augment -q fixed -N 2 -o /content/smoke_out") % (PRECISION, SEED, BATCH, LRATE),
   log=str(RUN / "smoke.log"), TORCHDYNAMO_DISABLE="1")
smoke_s = time.time() - t0
steps = -(-32 // BATCH) * 2
train_rows = sum(1 for _ in open(WORK / "kdata/train.txt"))
per_step = max(smoke_s - 60, 1) / steps   # subtract ~60 s of start-up/validation; conservative lower bound is fine
forecast_h = per_step * (train_rows / BATCH) * EPOCHS / 3600
print(f"smoke {smoke_s:.0f}s, ~{per_step:.1f}s/step (upper bound), forecast {forecast_h:.1f}h")
(RUN / "smoke.json").write_text(json.dumps({"smoke_seconds": smoke_s, "steps": steps, "forecast_hours_upper": forecast_h}))
assert forecast_h <= MAX_ESTIMATED_HOURS, "forecast over the gate: lower EPOCHS or stop (do not loosen the gate)"

In [ ]:
(RUN / "run_config.json").write_text(json.dumps(dict(
    run_id=RUN_ID, fold=FOLD, seed=SEED, epochs=EPOCHS, lrate=LRATE, batch=BATCH, precision=PRECISION,
    model="PP-OCRv6 medium (Zenodo 10.5281/zenodo.21788410)", model_license="Apache-2.0", weights_sha256=WEIGHTS_SHA,
    kraken="7.1.1", runtime="free Colab T4", provenance=SOURCE_PROVENANCE), indent=1))
t0 = time.time()
sh(("ketos -d cuda:0 --precision %s --workers 2 --threads 2 -s %d train --arch ppocrv6 --variant medium "
    "-i /content/medium.safetensors --resize union -f path -t kdata/train.txt -e kdata/validation.txt "
    "-B %d -r %g --optimizer AdamW -w 0.01 --schedule cosine --cos-max %d --warmup 100 --augment "
    "-q fixed -N %d -F %d -o /content/out") % (PRECISION, SEED, BATCH, LRATE, EPOCHS, EPOCHS, EPOCHS),
   log=str(RUN / "train.log"), TORCHDYNAMO_DISABLE="1")
(RUN / "train_minutes.txt").write_text("%.1f" % ((time.time() - t0) / 60))
weights = sorted(Path("/content/out").glob("best_*.safetensors"))
assert weights, "no weights written"
shutil.copy(weights[-1], RUN / "model.safetensors")
print("saved", RUN / "model.safetensors")

In [ ]:
model = RUN / "model.safetensors"
sh(f"python scripts/kraken_infer.py --input Train.csv --fold-manifest data/splits/folds.csv --fold {FOLD} "
   f"--model {model} --device cuda:0 --output {RUN}/fold{FOLD}_predictions.csv --metadata {RUN}/fold{FOLD}_metadata.json",
   log=str(RUN / "infer_fold.log"), TORCHDYNAMO_DISABLE="1")
score = json.loads((RUN / f"fold{FOLD}_metadata.json").read_text())["score"]
print("FOLD-%d  CER %.5f  WER %.5f  combined %.5f  (TrOCR control 0.15076; gate: lower)" % (FOLD, score["cer"], score["wer"], score["combined"]))
sh(f"python scripts/kraken_infer.py --input Test.csv --model {model} --device cuda:0 --output {RUN}/test_predictions.csv "
   f"--metadata {RUN}/test_metadata.json", log=str(RUN / "infer_test.log"), TORCHDYNAMO_DISABLE="1")
print("Artifacts on Drive:", RUN)